In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display

from src.data_utils import scan_audio_dataset
from src.features import EncodecFeatureExtractor, CLAPFeatureExtractor, extract_all_features
from src.analysis import (
    split_feature_sets,
    prepare_matrix,
    compute_pca,
    compute_umap,
    compute_silhouette,
)

In [ ]:
DATA_DIR = "data/raw"
OUTPUT_DIR = Path("data/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df = scan_audio_dataset(DATA_DIR)
df.head()

NameError: name 'DATA_DIR' is not defined

In [ ]:
print(df["label"].value_counts())

In [ ]:
encodec_extractor = EncodecFeatureExtractor()
clap_extractor = CLAPFeatureExtractor()

In [ ]:
rows = []

for i, row in df.iterrows():
    print(f"[{i+1}/{len(df)}] {row['filename']}")
    feats = extract_all_features(
        row["filepath"],
        encodec_extractor=encodec_extractor,
        clap_extractor=clap_extractor,
    )
    rows.append({**row.to_dict(), **feats})

feat_df = pd.DataFrame(rows)
feat_df.head()

In [ ]:
feat_df.to_csv(OUTPUT_DIR / "features.csv", index=False)
print("Saved:", OUTPUT_DIR / "features.csv")

In [ ]:
classical_cols, encodec_cols, clap_cols = split_feature_sets(feat_df)

print("Classical features:", len(classical_cols))
print("Encodec features:", len(encodec_cols))
print("CLAP features:", len(clap_cols))

In [ ]:
X_classical, _ = prepare_matrix(feat_df, classical_cols)
X_encodec, _ = prepare_matrix(feat_df, encodec_cols)
X_clap, _ = prepare_matrix(feat_df, clap_cols)

In [ ]:
Zc_umap, _ = compute_umap(X_classical)
Ze_umap, _ = compute_umap(X_encodec)
Zcl_umap, _ = compute_umap(X_clap)

In [ ]:
classical_sil = compute_silhouette(X_classical, feat_df["label"].values)
encodec_sil = compute_silhouette(X_encodec, feat_df["label"].values)
clap_sil = compute_silhouette(X_clap, feat_df["label"].values)

print("Silhouette score - Classical:", classical_sil)
print("Silhouette score - Encodec:", encodec_sil)
print("Silhouette score - CLAP:", clap_sil)

In [ ]:
plot_classical = feat_df.copy()
plot_classical["x"] = Zc_umap[:, 0]
plot_classical["y"] = Zc_umap[:, 1]

plt.figure(figsize=(8, 6))
for label, subdf in plot_classical.groupby("label"):
    plt.scatter(subdf["x"], subdf["y"], label=label, alpha=0.8)
plt.title("UMAP - Classical Audio Descriptors")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.show()

In [ ]:
plot_encodec = feat_df.copy()
plot_encodec["x"] = Ze_umap[:, 0]
plot_encodec["y"] = Ze_umap[:, 1]

plt.figure(figsize=(8, 6))
for label, subdf in plot_encodec.groupby("label"):
    plt.scatter(subdf["x"], subdf["y"], label=label, alpha=0.8)
plt.title("UMAP - Encodec Features")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.show()

In [ ]:
plot_clap = feat_df.copy()
plot_clap["x"] = Zcl_umap[:, 0]
plot_clap["y"] = Zcl_umap[:, 1]

plt.figure(figsize=(8, 6))
for label, subdf in plot_clap.groupby("label"):
    plt.scatter(subdf["x"], subdf["y"], label=label, alpha=0.8)
plt.title("UMAP - CLAP Embeddings")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.show()

In [ ]:
def show_audio_example(idx):
    row = feat_df.iloc[idx]
    print("Filename:", row["filename"])
    print("Label:", row["label"])
    print("Path:", row["filepath"])

    y, sr = librosa.load(row["filepath"], sr=None, mono=True)

    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(row["filename"])
    plt.show()

    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="log")
    plt.colorbar(format="%+2.0f dB")
    plt.title("Spectrogram")
    plt.show()

    display(Audio(row["filepath"]))

In [ ]:
show_audio_example(0)

In [ ]:
feat_df[[
    "filename",
    "label",
    "rms_mean",
    "zcr_mean",
    "spectral_centroid_mean",
    "spectral_bandwidth_mean",
    "spectral_rolloff_mean",
    "onset_strength_mean",
    "encodec_num_codebooks",
    "encodec_total_frames",
    "clap_dim"
]].head(10)

In [ ]:
summary_cols = [
    "rms_mean",
    "zcr_mean",
    "spectral_centroid_mean",
    "spectral_bandwidth_mean",
    "spectral_rolloff_mean",
    "onset_strength_mean",
]

summary_df = feat_df.groupby("label")[summary_cols].mean().round(3)
summary_df

In [ ]:
scores_df = pd.DataFrame([
    {"feature_set": "classical", "silhouette_score": classical_sil},
    {"feature_set": "encodec", "silhouette_score": encodec_sil},
    {"feature_set": "clap", "silhouette_score": clap_sil},
])

scores_df

In [ ]:
scores_df.to_csv(OUTPUT_DIR / "silhouette_scores.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "category_summary.csv")
print("Saved results tables.")